# Notebook 3: Evaluasi Agen DQN dan Tabel Q-Value

**Proyek** : Dynamic Pricing untuk Asuransi Kesehatan

**Tujuan Notebook Ini** :
1. Memuat model DQN yang telah terlatih dari `models/dqn_insurance_pricing.zip`.
2. Membandingkan profitabilitas agen DQN terhadap strategi penetapan harga statis (baseline).
3. Menghasilkan dan menginterpretasikan tabel Q-value.
4. Memvisualisasikan distribusi profit dan distribusi aksi yang dipilih agen.

---

> **Prasyarat**: Jalankan `02_pelatihan_dqn.ipynb` terlebih dahulu agar model tersedia di folder `models/`.

## 1. Import Library

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from sklearn.preprocessing import StandardScaler, LabelEncoder
from stable_baselines3 import DQN

print("Library berhasil diimpor.")

## 2. Konfigurasi

In [ ]:
DATASET_PATH = "../medical_insurance.csv"
MODEL_PATH   = "./models/dqn_insurance_pricing"
ARTIFACT_DIR = "./artifacts"

os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Label aksi untuk visualisasi
ACTION_MULTIPLIERS = np.array([0.8, 0.9, 1.0, 1.1, 1.2])
ACTION_LABELS      = [
    "0.8x (Diskon Besar)",
    "0.9x (Diskon Kecil)",
    "1.0x (Standar)",
    "1.1x (Naik Kecil)",
    "1.2x (Naik Besar)",
]

print(f"Model path  : {os.path.abspath(MODEL_PATH)}.zip")
print(f"Dataset     : {os.path.abspath(DATASET_PATH)}")

## 3. Preprocessing Data dan Environment (Self-Contained)

In [ ]:
def load_and_preprocess(filepath: str):
    """Memuat dan memproses dataset asuransi."""
    df = pd.read_csv(filepath)
    for col in ["gender", "discount_eligibility", "region"]:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    feature_cols = ["age", "gender", "bmi", "children", "discount_eligibility", "region"]
    numeric_cols = ["age", "bmi", "children"]
    scaler = StandardScaler()
    df_features = df[feature_cols].copy()
    df_features[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    return (
        df_features.values.astype(np.float32),
        df["expenses"].values.astype(np.float32),
        df["premium"].values.astype(np.float32),
    )


class InsurancePricingEnv(gym.Env):
    """Custom Gymnasium Environment (self-contained untuk notebook evaluasi)."""
    metadata = {"render_modes": ["human"]}
    ACTION_MULTIPLIERS = np.array([0.8, 0.9, 1.0, 1.1, 1.2], dtype=np.float32)

    def __init__(self, states, expenses, premiums, penalty_threshold=1.5, penalty_coef=2.0):
        super().__init__()
        self.states = states
        self.expenses = expenses
        self.premiums = premiums
        self.n_samples = len(states)
        self.penalty_threshold = penalty_threshold
        self.penalty_coef = penalty_coef
        self.action_space = spaces.Discrete(len(self.ACTION_MULTIPLIERS))
        self.observation_space = spaces.Box(-np.inf, np.inf, (states.shape[1],), np.float32)
        self._current_idx = 0
        self._last_info = {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._current_idx = 0
        return self.states[0], {}

    def step(self, action):
        idx = self._current_idx
        base_premium = float(self.premiums[idx])
        actual_expenses = float(self.expenses[idx])
        multiplier = float(self.ACTION_MULTIPLIERS[action])
        proposed = base_premium * multiplier
        profit = proposed - actual_expenses
        excess = proposed - self.penalty_threshold * actual_expenses
        penalty = self.penalty_coef * max(0.0, excess)
        reward = float(np.clip((profit - penalty) / 1000.0, -10.0, 10.0))
        self._last_info = {
            "index": idx, "base_premium": base_premium, "multiplier": multiplier,
            "proposed_premium": proposed, "actual_expenses": actual_expenses,
            "profit": profit, "penalty": penalty, "raw_reward": profit - penalty,
        }
        self._current_idx += 1
        terminated = self._current_idx >= self.n_samples
        observation = self.states[idx] if terminated else self.states[self._current_idx]
        return observation, reward, terminated, False, self._last_info

    def render(self): pass
    def close(self): pass


# Muat data dan buat environment
states, expenses, premiums = load_and_preprocess(DATASET_PATH)
env = InsurancePricingEnv(states=states, expenses=expenses, premiums=premiums)
print(f"Data dimuat: {states.shape[0]} sampel, {states.shape[1]} fitur.")

## 4. Memuat Model DQN yang Terlatih

In [ ]:
if not os.path.exists(MODEL_PATH + ".zip"):
    raise FileNotFoundError(
        f"Model tidak ditemukan: {MODEL_PATH}.zip\n"
        "Jalankan notebook 02_pelatihan_dqn.ipynb terlebih dahulu."
    )

model = DQN.load(MODEL_PATH, env=env)
print(f"Model berhasil dimuat dari: {MODEL_PATH}.zip")
print(f"Total parameter jaringan Q: {sum(p.numel() for p in model.q_net.parameters()):,}")

## 5. Evaluasi Agen DQN

Agen DQN dijalankan pada seluruh dataset. Untuk setiap pasien, agen memilih aksi secara deterministik (tanpa eksplorasi) berdasarkan argmax Q-value.

In [ ]:
def evaluate_agent(model, env: InsurancePricingEnv, deterministic: bool = True) -> pd.DataFrame:
    """
    Menjalankan agen pada seluruh dataset dan mencatat hasil per langkah.

    Parameters
    ----------
    model         : DQN  -- model yang telah terlatih
    env           : InsurancePricingEnv
    deterministic : bool -- jika True, agen menggunakan argmax Q-value

    Returns
    -------
    pd.DataFrame berisi hasil per pasien
    """
    obs, _ = env.reset()
    records = []

    while True:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(int(action))
        records.append({
            "index"           : info["index"],
            "action"          : int(action),
            "multiplier"      : info["multiplier"],
            "base_premium"    : info["base_premium"],
            "proposed_premium": info["proposed_premium"],
            "actual_expenses" : info["actual_expenses"],
            "profit"          : info["profit"],
            "penalty"         : info["penalty"],
            "net_reward"      : info["raw_reward"],
        })
        if terminated or truncated:
            break

    return pd.DataFrame(records)


def evaluate_static_baseline(env: InsurancePricingEnv, fixed_action: int = 2) -> pd.DataFrame:
    """
    Mengevaluasi strategi statis: agen selalu memilih aksi yang sama.
    Default aksi 2 = pengali 1.0x (harga standar).
    """
    obs, _ = env.reset()
    records = []

    while True:
        obs, reward, terminated, truncated, info = env.step(fixed_action)
        records.append({
            "index"           : info["index"],
            "action"          : fixed_action,
            "multiplier"      : info["multiplier"],
            "base_premium"    : info["base_premium"],
            "proposed_premium": info["proposed_premium"],
            "actual_expenses" : info["actual_expenses"],
            "profit"          : info["profit"],
            "penalty"         : info["penalty"],
            "net_reward"      : info["raw_reward"],
        })
        if terminated or truncated:
            break

    return pd.DataFrame(records)


print("Menjalankan evaluasi agen DQN...")
dqn_results      = evaluate_agent(model, env)

print("Menjalankan evaluasi strategi statis (1.0x)...")
baseline_results = evaluate_static_baseline(env, fixed_action=2)

print(f"Evaluasi selesai: {len(dqn_results)} pasien diproses.")
dqn_results.head()

## 6. Ringkasan Metrik Profitabilitas

In [ ]:
metrics = [
    ("Total Profit",      dqn_results["profit"].sum(),      baseline_results["profit"].sum()),
    ("Rata-rata Profit",  dqn_results["profit"].mean(),     baseline_results["profit"].mean()),
    ("Median Profit",     dqn_results["profit"].median(),   baseline_results["profit"].median()),
    ("Std Profit",        dqn_results["profit"].std(),      baseline_results["profit"].std()),
    ("Total Penalti",     dqn_results["penalty"].sum(),     baseline_results["penalty"].sum()),
    ("Total Net Reward",  dqn_results["net_reward"].sum(),  baseline_results["net_reward"].sum()),
]

summary_df = pd.DataFrame(
    metrics,
    columns=["Metrik", "Agen DQN", "Statis (1.0x)"],
)
summary_df["Selisih"] = summary_df["Agen DQN"] - summary_df["Statis (1.0x)"]
summary_df["Selisih (%)"] = (
    summary_df["Selisih"] / summary_df["Statis (1.0x)"].abs() * 100
).round(2)

# Format angka
for col in ["Agen DQN", "Statis (1.0x)", "Selisih"]:
    summary_df[col] = summary_df[col].map(lambda x: f"{x:,.2f}")

print("\nRingkasan Metrik Profitabilitas:")
print(summary_df.to_string(index=False))

## 7. Visualisasi Perbandingan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel Kiri: Distribusi Profit ---
axes[0].hist(baseline_results["profit"], bins=40, alpha=0.6,
             color="#EF5350", label="Statis (1.0x)",
             edgecolor="white", linewidth=0.3)
axes[0].hist(dqn_results["profit"], bins=40, alpha=0.6,
             color="#42A5F5", label="Agen DQN",
             edgecolor="white", linewidth=0.3)
axes[0].axvline(dqn_results["profit"].mean(), color="#1565C0",
                linestyle="--", linewidth=2,
                label=f"Mean DQN: {dqn_results['profit'].mean():,.1f}")
axes[0].axvline(baseline_results["profit"].mean(), color="#B71C1C",
                linestyle="--", linewidth=2,
                label=f"Mean Statis: {baseline_results['profit'].mean():,.1f}")
axes[0].set_xlabel("Profit (USD)", fontsize=11)
axes[0].set_ylabel("Frekuensi", fontsize=11)
axes[0].set_title("Distribusi Profit per Pasien", fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle="--", alpha=0.4)

# --- Panel Kanan: Distribusi Aksi DQN ---
action_counts = dqn_results["action"].value_counts().sort_index()
bar_colors    = ["#EF5350", "#FF7043", "#66BB6A", "#42A5F5", "#5C6BC0"]
bars = axes[1].bar(
    [ACTION_LABELS[i] for i in action_counts.index],
    action_counts.values,
    color=[bar_colors[i] for i in action_counts.index],
    edgecolor="white", linewidth=0.5,
)
for bar in bars:
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(int(bar.get_height())),
        ha="center", va="bottom", fontsize=9,
    )
axes[1].set_xlabel("Aksi (Pengali Premi)", fontsize=11)
axes[1].set_ylabel("Jumlah Pasien", fontsize=11)
axes[1].set_title("Distribusi Aksi yang Dipilih Agen DQN", fontsize=13)
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(True, linestyle="--", alpha=0.4, axis="y")

plt.tight_layout()
comparison_path = os.path.join(ARTIFACT_DIR, "comparison_plot.png")
plt.savefig(comparison_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Plot disimpan ke: {comparison_path}")

## 8. Tabel Q-Value

### 8.1 Teori Q-Value

**Q-value** $Q(s, a)$ merepresentasikan estimasi total return kumulatif yang didiskonto jika agen berada di state $s$, mengambil aksi $a$, dan selanjutnya mengikuti kebijakan optimal:

$$Q^*(s, a) = \mathbb{E}\left[ r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \ldots \mid s_t = s,\ a_t = a \right]$$

Kebijakan optimal agen adalah:

$$\pi^*(s) = \arg\max_a Q^*(s, a)$$

### 8.2 Pembaruan Q (Bellman Equation)

$$Q(s, a) \leftarrow Q(s, a) + \alpha \underbrace{\left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]}_{\text{TD Error } \delta}$$

### 8.3 Loss DQN

$$\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[ \left( \underbrace{r + \gamma \max_{a'} Q_{\theta^-}(s', a')}_{\text{target } y} - Q_{\theta}(s, a) \right)^2 \right]$$

In [ ]:
def build_q_value_table(model, env: InsurancePricingEnv, n_samples: int = 20) -> pd.DataFrame:
    """
    Mengekstrak Q-value untuk N sampel yang tersebar merata di dataset.

    Karena DQN menggunakan jaringan neural, Q-value dihitung dengan
    melewatkan observasi melalui Q-network (forward pass, tanpa gradient).

    Parameters
    ----------
    model     : DQN
    env       : InsurancePricingEnv
    n_samples : int -- jumlah sampel yang diambil

    Returns
    -------
    pd.DataFrame berisi Q-value tiap aksi untuk setiap sampel
    """
    records = []
    indices = np.linspace(0, env.n_samples - 1, n_samples, dtype=int)

    for idx in indices:
        state      = env.states[idx]
        obs_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)

        # Forward pass melalui Q-network (tanpa komputasi gradient)
        with torch.no_grad():
            q_values = model.q_net(obs_tensor).numpy().flatten()

        best_action = int(np.argmax(q_values))

        record = {"Indeks Sampel": idx}
        for a, label in enumerate(ACTION_LABELS):
            record[f"Q({label})"] = round(float(q_values[a]), 5)
        record["Aksi Terbaik (argmax Q)"] = ACTION_LABELS[best_action]
        records.append(record)

    return pd.DataFrame(records)


print("Membangun tabel Q-Value (20 sampel)...")
q_table = build_q_value_table(model, env, n_samples=20)
print("Tabel Q-Value berhasil dibangun.")
q_table

## 9. Visualisasi Q-Value (Heatmap)

In [ ]:
# Ekstrak kolom Q-value untuk heatmap
q_cols    = [col for col in q_table.columns if col.startswith("Q(")]
q_matrix  = q_table[q_cols].values
short_labels = ["0.8x", "0.9x", "1.0x", "1.1x", "1.2x"]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(q_matrix, cmap="RdYlGn", aspect="auto")

# Annotasi nilai
for i in range(q_matrix.shape[0]):
    for j in range(q_matrix.shape[1]):
        val = q_matrix[i, j]
        color = "black" if abs(val) < (q_matrix.max() - q_matrix.min()) * 0.6 + q_matrix.min() else "white"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=7.5)

ax.set_xticks(range(5))
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_yticks(range(len(q_table)))
ax.set_yticklabels([f"Sampel {int(i)}" for i in q_table["Indeks Sampel"]], fontsize=8)
ax.set_xlabel("Aksi (Pengali Premi)", fontsize=11)
ax.set_ylabel("Indeks Sampel Pasien", fontsize=11)
ax.set_title("Heatmap Q-Value: Nilai Estimasi per Aksi per Pasien\n(Hijau = Q-value Tinggi, Merah = Q-value Rendah)", fontsize=12)
plt.colorbar(im, ax=ax, label="Q-Value")

plt.tight_layout()
qtable_path = os.path.join(ARTIFACT_DIR, "q_value_heatmap.png")
plt.savefig(qtable_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Heatmap Q-Value disimpan ke: {qtable_path}")

## 10. Distribusi Aksi Terbaik dalam Q-Table

In [ ]:
# Hitung frekuensi aksi terbaik dalam sampel Q-table
best_action_counts = q_table["Aksi Terbaik (argmax Q)"].value_counts()

print("Distribusi Aksi Terbaik (dari 20 sampel):")
for action_label, count in best_action_counts.items():
    pct = count / len(q_table) * 100
    bar = "#" * int(pct / 5)
    print(f"  {action_label:25s}: {count:3d} ({pct:5.1f}%) {bar}")

## 11. Menyimpan Hasil Evaluasi

In [ ]:
# Simpan seluruh hasil ke CSV
dqn_csv      = os.path.join(ARTIFACT_DIR, "dqn_evaluation_results.csv")
baseline_csv = os.path.join(ARTIFACT_DIR, "baseline_evaluation_results.csv")
qtable_csv   = os.path.join(ARTIFACT_DIR, "q_value_table.csv")

dqn_results.to_csv(dqn_csv, index=False)
baseline_results.to_csv(baseline_csv, index=False)
q_table.to_csv(qtable_csv, index=False)

print("Hasil evaluasi berhasil disimpan:")
print(f"  {dqn_csv}")
print(f"  {baseline_csv}")
print(f"  {qtable_csv}")

## 12. Scatter Plot: Proposed Premium vs Actual Expenses

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df_plot, title, color in [
    (axes[0], dqn_results,      "Agen DQN",       "#42A5F5"),
    (axes[1], baseline_results, "Statis (1.0x)",  "#EF5350"),
]:
    ax.scatter(
        df_plot["actual_expenses"],
        df_plot["proposed_premium"],
        alpha=0.4, s=15, color=color, edgecolors="none",
    )
    # Garis referensi y=x (break-even)
    max_val = max(df_plot["actual_expenses"].max(), df_plot["proposed_premium"].max())
    ax.plot([0, max_val], [0, max_val], "k--", linewidth=1.2, label="Break-even (y=x)")
    ax.set_xlabel("Actual Expenses (USD)", fontsize=11)
    ax.set_ylabel("Proposed Premium (USD)", fontsize=11)
    ax.set_title(f"Proposed Premium vs Actual Expenses\n({title})", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
scatter_path = os.path.join(ARTIFACT_DIR, "scatter_premium_vs_expenses.png")
plt.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Scatter plot disimpan ke: {scatter_path}")

## 13. Ringkasan Akhir

Notebook ini telah menyelesaikan:

1. Memuat model DQN yang telah terlatih dari `models/dqn_insurance_pricing.zip`.
2. Menjalankan evaluasi deterministik agen DQN pada 1.338 pasien.
3. Mengevaluasi strategi statis (harga standar 1.0x) sebagai pembanding.
4. Membandingkan metrik profitabilitas (total profit, rata-rata profit, penalti, net reward).
5. Membangun tabel Q-value (20 sampel) menggunakan forward pass Q-network.
6. Memvisualisasikan heatmap Q-value dan distribusi aksi.
7. Menyimpan semua artefak ke folder `artifacts/`.

---

### Artefak yang Dihasilkan

| File | Keterangan |
|------|------------|
| `artifacts/dqn_evaluation_results.csv` | Hasil evaluasi per pasien (DQN) |
| `artifacts/baseline_evaluation_results.csv` | Hasil evaluasi strategi statis |
| `artifacts/q_value_table.csv` | Tabel Q-value 20 sampel |
| `artifacts/comparison_plot.png` | Plot histogram profit dan distribusi aksi |
| `artifacts/q_value_heatmap.png` | Heatmap Q-value per aksi per sampel |
| `artifacts/scatter_premium_vs_expenses.png` | Scatter plot proposed premium vs actual expenses |